In [1]:
# Importa as bibliotecas

import os
from dotenv import load_dotenv

import pandas as pd
import psycopg2 as pg
import sqlalchemy
from sqlalchemy import create_engine
import panel as pn

In [2]:
# Carrega as variáveis do arquivo .env

load_dotenv()

True

In [3]:
# Lê as variáveis de ambiente

DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')

In [4]:
# Cria conexão com psycopg2 usando as variáveis carregadas

con = pg.connect(host=DB_HOST, dbname=DB_NAME, user=DB_USER, password=DB_PASS)

In [5]:
# Define a string de conexão para o SQLAlchemy, utilizando as variáveis do .env
# Cria o objeto engine do SQLAlchemy que será usado para conectar e executar comandos no banco

cnx = f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}'

engine = create_engine(cnx)

In [6]:
# Executa a consulta SQL para buscar todos os 
# registros da tabela 'pessoa' no banco PostgreSQL 
# e carrega o resultado em um DataFrame do pandas


query = "select * from usuario;" 
df = pd.read_sql_query(query, engine)


In [7]:
pn.extension('tabulator')

In [8]:
# Inputs
cpf_usuario = pn.widgets.TextInput(name="CPF", placeholder="Digite o CPF")
nome_usuario = pn.widgets.TextInput(name="Nome", placeholder="Digite o nome")
genero_usuario = pn.widgets.Select(name="Gênero", options=["M", "F", "O"])
telefone_usuario = pn.widgets.TextInput(name="Telefone", placeholder="Digite o telefone")
data_nasc_usuario = pn.widgets.DatePicker(name="Data de Nascimento")
endereco_usuario = pn.widgets.TextInput(name="Endereço", placeholder="Digite o endereço")
email_usuario = pn.widgets.TextInput(name="Email", placeholder="Digite o email")
tipo_sanguineo_usuario = pn.widgets.Select(
    name="Tipo Sanguíneo",
    options=["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"]
)
tipo_usuario = pn.widgets.RadioButtonGroup(
    name='Tipo de Usuário', 
    options=['Doador', 'Receptor'], 
    button_type='success'
)

btn_consultar_user = pn.widgets.Button(name="Consultar", button_type="primary", width=85)
btn_inserir_user = pn.widgets.Button(name="Inserir", button_type="success", width=85)
btn_editar_user = pn.widgets.Button(name="Editar", button_type="warning", width=85)
btn_excluir_user = pn.widgets.Button(name="Excluir", button_type="danger", width=85)


In [9]:
# Funções de ação
def queryAllUsuarios():
    try:
        query = "SELECT * FROM usuario"
        df = pd.read_sql_query(query, engine)
        return pn.widgets.Tabulator(df, pagination='remote', page_size=10)
    except:
        return pn.pane.Alert("Erro ao consultar dados!")

def on_consultar_usuario():
    try:
        query = f"SELECT * FROM usuario WHERE cpf = '{cpf_usuario.value}'"
        df = pd.read_sql_query(query, engine)
        return pn.widgets.Tabulator(df)
    except:
        return pn.pane.Alert("Erro na consulta!")

def on_inserir_usuario():
    try:
        cursor = con.cursor()

        # Verifica se o CPF já existe em usuario
        cursor.execute("SELECT count(*) FROM usuario WHERE cpf = %s", (cpf_usuario.value,))

        # Insere na tabela usuario
        cursor.execute("""
            INSERT INTO usuario (cpf, nome, genero, telefone, data_nasc, endereco, email, tipo_sanguineo)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            cpf_usuario.value, nome_usuario.value, genero_usuario.value,
            telefone_usuario.value, data_nasc_usuario.value,
            endereco_usuario.value, email_usuario.value, tipo_sanguineo_usuario.value
        ))

        # Insere como DOADOR ou RECEPTOR (exclusivo)
        if tipo_usuario.value == 'Doador':
            cursor.execute("INSERT INTO doador (cpf_usuario) VALUES (%s)", (cpf_usuario.value,))
        else:
            cursor.execute("INSERT INTO receptor (cpf_usuario) VALUES (%s)", (cpf_usuario.value,))

        con.commit()
        return pn.pane.Alert("Usuário cadastrado com sucesso!", alert_type="success")
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f"Erro ao inserir: {str(e)}", alert_type="danger")

def on_excluir_usuario():
    try:
        cursor = con.cursor()
        
        # Primeiro verifica se o usuário existe
        cursor.execute("SELECT count(*) FROM usuario WHERE cpf = %s", (cpf_usuario.value,))
        if cursor.fetchone()[0] == 0:
            return pn.pane.Alert("CPF não encontrado.", alert_type="warning")
        
        # Remove das tabelas dependentes primeiro
        cursor.execute("DELETE FROM doador WHERE cpf_usuario = %s", (cpf_usuario.value,))
        cursor.execute("DELETE FROM receptor WHERE cpf_usuario = %s", (cpf_usuario.value,))
        
        # Agora remove do usuário
        cursor.execute("DELETE FROM usuario WHERE cpf = %s", (cpf_usuario.value,))
        
        con.commit()
        
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f"Erro ao excluir usuário: {str(e)}", alert_type="danger")
def on_editar_usuario():
    try:
        cursor = con.cursor()
        
        # Verifica se o usuário existe
        cursor.execute("SELECT count(*) FROM usuario WHERE cpf = %s", (cpf_usuario.value,))
        
        # Atualiza os dados do usuário
        cursor.execute("""
            UPDATE usuario SET
                nome = %s,
                genero = %s,
                telefone = %s,
                data_nasc = %s,
                endereco = %s,
                email = %s,
                tipo_sanguineo = %s
            WHERE cpf = %s
        """, (
            nome_usuario.value,
            genero_usuario.value,
            telefone_usuario.value,
            data_nasc_usuario.value,
            endereco_usuario.value,
            email_usuario.value,
            tipo_sanguineo_usuario.value,
            cpf_usuario.value
        ))

        # Remove de doador/receptor se o tipo foi alterado
        cursor.execute("DELETE FROM doador WHERE cpf_usuario = %s", (cpf_usuario.value,))
        cursor.execute("DELETE FROM receptor WHERE cpf_usuario = %s", (cpf_usuario.value,))

        # Insere no novo tipo selecionado
        if tipo_usuario.value == 'Doador':
            cursor.execute("INSERT INTO doador (cpf_usuario) VALUES (%s)", (cpf_usuario.value,))
        else:
            cursor.execute("INSERT INTO receptor (cpf_usuario) VALUES (%s)", (cpf_usuario.value,))

        con.commit()
        return pn.pane.Alert("Dados atualizados com sucesso!", alert_type="success")
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f"Erro ao editar: {str(e)}", alert_type="danger")
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f"Erro ao editar: {e}")

In [10]:
# Lógica interativa com a biblioteca Panel
def render_usuario(consultar, inserir, excluir, editar):
    if consultar:
        return on_consultar_usuario()
    elif inserir:
        return on_inserir_usuario()
    elif excluir:
        return on_excluir_usuario()
    elif editar:
        return on_editar_usuario()
    else:
        return queryAllUsuarios()
tabela_usuario = pn.bind(render_usuario, btn_consultar_user, btn_inserir_user, btn_excluir_user, btn_editar_user)


In [11]:
## Interface
pn.Row(
    pn.Column(
        "### Cadastro de Usuários",
        cpf_usuario, nome_usuario, genero_usuario, telefone_usuario, data_nasc_usuario,
        endereco_usuario, email_usuario, tipo_sanguineo_usuario, tipo_usuario,
        pn.Row(btn_consultar_user, btn_inserir_user, btn_editar_user, btn_excluir_user),
    ),
    pn.Column(tabela_usuario)
).servable()

Row
    [0] Column
        [0] Markdown(str)
        [1] TextInput(name='CPF', placeholder='Digite o CPF')
        [2] TextInput(name='Nome', placeholder='Digite o nome')
        [3] Select(name='Gênero', options=['M', 'F', 'O'], value='M')
        [4] TextInput(name='Telefone', placeholder='Digite o telefone')
        [5] DatePicker(name='Data de Nascimento')
        [6] TextInput(name='Endereço', placeholder='Digite o endereço')
        [7] TextInput(name='Email', placeholder='Digite o email')
        [8] Select(name='Tipo Sanguíneo', options=['A+', 'A-', 'B+', ...], value='A+')
        [9] RadioButtonGroup(button_type='success', name='Tipo de Usuário', options=['Doador', 'Receptor'], value='Doador')
        [10] Row
            [0] Button(button_type='primary', name='Consultar', width=85)
            [1] Button(button_type='success', name='Inserir', width=85)
            [2] Button(button_type='warning', name='Editar', width=85)
            [3] Button(button_type='danger', name='Excluir', width=85)
    [1] Column
        [0] ParamFunction(function, _pane=Tabulator, defer_load=False)

In [12]:
#Input
nome_instituicao_local = pn.widgets.TextInput(name="Nome da Instituição", placeholder="Digite o nome da instituição")
endereco_local = pn.widgets.TextInput(name="Endereço", placeholder="Digite o endereço")
telefone_local = pn.widgets.TextInput(name="Telefone", placeholder="Digite o telefone")
horario_local = pn.widgets.TextInput(name="Horário", placeholder="HH:MM")

In [13]:
# Botões
btn_consultar_local = pn.widgets.Button(name="Consultar", button_type="primary", width=85)
btn_inserir_local = pn.widgets.Button(name="Inserir", button_type="success", width=85)
btn_editar_local = pn.widgets.Button(name="Editar", button_type="warning", width=85)
btn_excluir_local = pn.widgets.Button(name="Excluir", button_type="danger", width=85)

In [14]:
import pandas as pd
from datetime import datetime

def queryAllLocais():
    try:
        query = "SELECT id, nome_instituicao, endereco, telefone, horario FROM local"
        df = pd.read_sql_query(query, engine)

        if 'horario' in df.columns and not df['horario'].isnull().all():
            df['horario'] = df['horario'].astype(str).str[:5]

        return pn.widgets.Tabulator(df, pagination='remote', page_size=10, show_index=False)
    except Exception as e:
        return pn.pane.Alert(f"Erro ao consultar locais: {e}", alert_type="danger")
    
def on_consultar_local(event=None):
    try:
        query = "SELECT nome_instituicao, endereco, telefone, horario FROM local WHERE nome_instituicao = %s"
        df = pd.read_sql_query(query, engine, params=(nome_instituicao_local.value,))
        
        if df.empty:
            return pn.pane.Alert("Instituição não encontrada.", alert_type="warning")
        
        # Formata o horário
        if 'horario' in df.columns:
            df['horario'] = df['horario'].astype(str).str[:5]
            
        return pn.widgets.Tabulator(df, show_index=False)
    except Exception as e:
        return pn.pane.Alert(f"Erro na consulta: {str(e)}", alert_type="danger")
    
def on_inserir_local(event=None):
    try:
        cursor = con.cursor()

        # Verifica se a instituição já existe
        cursor.execute("SELECT count(*) FROM local WHERE nome_instituicao = %s", (nome_instituicao_local.value,))
        if cursor.fetchone()[0] > 0:
            return pn.pane.Alert("Instituição já cadastrada!", alert_type="warning")

        # Converte horário
        try:
            horario = datetime.strptime(horario_local.value, "%H:%M").time()
        except:
            return pn.pane.Alert("Formato de horário inválido! Use HH:MM", alert_type="warning")

        # Insere o local
        cursor.execute("""
            INSERT INTO local (nome_instituicao, endereco, telefone, horario)
            VALUES (%s, %s, %s, %s)
        """, (
            nome_instituicao_local.value,
            endereco_local.value,
            telefone_local.value,
            horario
        ))

        con.commit()
        return pn.pane.Alert("Local cadastrado com sucesso!", alert_type="success")
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f"Erro ao inserir local: {str(e)}", alert_type="danger")
    

def on_excluir_local(event=None):
    try:
        cursor = con.cursor()
        cursor.execute("DELETE FROM local WHERE nome_instituicao = %s", (nome_instituicao_local.value,))
        con.commit()
        if cursor.rowcount > 0:
            return queryAllLocais()
        else:
            return pn.pane.Alert("Instituição não encontrada.", alert_type="warning")
    except:
        con.rollback()
        return pn.pane.Alert("Erro ao excluir local!", alert_type="danger")

def on_editar_local(event=None):
    try:
        cursor = con.cursor()

        # Verifica se o local existe
        cursor.execute("SELECT count(*) FROM local WHERE nome_instituicao = %s", (nome_instituicao_local.value,))
        if cursor.fetchone()[0] == 0:
            return pn.pane.Alert("Instituição não encontrada!", alert_type="warning")

        # Converte horário
        try:
            horario = datetime.strptime(horario_local.value, "%H:%M").time()
        except:
            return pn.pane.Alert("Formato de horário inválido! Use HH:MM", alert_type="warning")

        # Atualiza os dados
        cursor.execute("""
            UPDATE local SET
                horario = %s,
                endereco = %s,
                telefone = %s
            WHERE nome_instituicao = %s
        """, (
            horario,
            endereco_local.value,
            telefone_local.value,
            nome_instituicao_local.value
        ))

        con.commit()
        return pn.pane.Alert("Local atualizado com sucesso!", alert_type="success")
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f"Erro ao editar local: {str(e)}", alert_type="danger")

In [15]:
# Lógica interativa com a biblioteca Panel
def render_local(consultar, inserir, excluir, editar):
    if consultar:
        return on_consultar_local()
    elif inserir:
        return on_inserir_local()
    elif excluir:
        return on_excluir_local()
    elif editar:
        return on_editar_local()
    else:
        return queryAllLocais()

In [16]:
# Vinculando a função aos botões
tabela_local = pn.bind(
    render_local,
    btn_consultar_local,
    btn_inserir_local,
    btn_excluir_local,
    btn_editar_local
)

In [17]:
# Interface
pn.Row(
    pn.Column(
        "### Cadastro de Local",
          nome_instituicao_local, telefone_local, endereco_local, horario_local,
        pn.Row(btn_consultar_local, btn_inserir_local, btn_editar_local, btn_excluir_local),
    ),
    pn.Column(tabela_local)
).servable()

Row
    [0] Column
        [0] Markdown(str)
        [1] TextInput(name='Nome da Instituição', placeholder='Digite o nome d...)
        [2] TextInput(name='Telefone', placeholder='Digite o telefone')
        [3] TextInput(name='Endereço', placeholder='Digite o endereço')
        [4] TextInput(name='Horário', placeholder='HH:MM')
        [5] Row
            [0] Button(button_type='primary', name='Consultar', width=85)
            [1] Button(button_type='success', name='Inserir', width=85)
            [2] Button(button_type='warning', name='Editar', width=85)
            [3] Button(button_type='danger', name='Excluir', width=85)
    [1] Column
        [0] ParamFunction(function, _pane=Tabulator, defer_load=False)

In [ ]:
cod_bolsa = pn.widgets.IntInput(name="Código da Bolsa", placeholder="Digite o código...")
cpf_doador_bolsa = pn.widgets.TextInput(name="CPF do Doador", placeholder="Digite o CPF...")
data_coleta_bolsa = pn.widgets.DatePicker(name="Data da Coleta")
data_validade_bolsa = pn.widgets.DatePicker(name="Data de Validade")
id_agendamento_bolsa = pn.widgets.IntInput(name="ID do Agendamento", placeholder="Digite o ID...")
tipo_sanguineo_bolsa = pn.widgets.Select(
    name="Tipo Sanguíneo",
    options=["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"]
)
status_bolsa = pn.widgets.Select(
    name="Status da Bolsa",
    options=['Disponível', 'Utilizada', 'Vencida', 'Reservada']
)



btn_consultar_bolsa = pn.widgets.Button(name="Consultar", button_type="primary", width=85)
btn_inserir_bolsa = pn.widgets.Button(name="Inserir", button_type="success", width=85)
btn_editar_bolsa = pn.widgets.Button(name="Editar", button_type="warning", width=85)
btn_excluir_bolsa = pn.widgets.Button(name="Excluir", button_type="danger", width=85)




def queryAllBolsas():
    try:
        query = "SELECT * FROM bolsa ORDER BY cod_bolsa"
        df = pd.read_sql_query(query, engine)
        
        formatador_bolsa = {'cod_bolsa': {'type': 'number', 'thousand': ''}}
        
        return pn.widgets.Tabulator(df, pagination='remote', page_size=10, show_index=False, formatters=formatador_bolsa)
    except Exception as e:
        return pn.pane.Alert(f"Erro ao consultar bolsas: {e}", alert_type="danger")

def on_inserir_bolsa():
    if not all([cod_bolsa.value, cpf_doador_bolsa.value, data_coleta_bolsa.value, tipo_sanguineo_bolsa.value, status_bolsa.value]):
        return pn.pane.Alert("Preencha todos os campos obrigatórios para inserir!", alert_type="warning")
    try:
        cursor = con.cursor()
        cursor.execute("""
            INSERT INTO bolsa (cod_bolsa, cpf_doador, data_coleta, data_validade, id_agendamento, tipo_sanguineo, status)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (
            cod_bolsa.value, cpf_doador_bolsa.value, data_coleta_bolsa.value,
            data_validade_bolsa.value, id_agendamento_bolsa.value,
            tipo_sanguineo_bolsa.value, status_bolsa.value
        ))
        con.commit()
        return pn.pane.Alert("Bolsa cadastrada com sucesso!", alert_type="success")
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f"Erro ao inserir bolsa: {str(e)}", alert_type="danger")

def on_excluir_bolsa():
    if not cod_bolsa.value:
        return pn.pane.Alert("Digite o Código da Bolsa para excluir.", alert_type="warning")
    try:
        cursor = con.cursor()
        cursor.execute("DELETE FROM bolsa WHERE cod_bolsa = %s", (cod_bolsa.value,))
        con.commit()
        if cursor.rowcount > 0:
            return pn.pane.Alert("Bolsa excluída com sucesso!", alert_type="success")
        else:
            return pn.pane.Alert("Código da bolsa não encontrado.", alert_type="warning")
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f"Erro ao excluir: {str(e)}", alert_type="danger")

def on_editar_bolsa():
    if not cod_bolsa.value:
        return pn.pane.Alert("Digite o Código da Bolsa para editar.", alert_type="warning")
    try:
        cursor = con.cursor()
        cursor.execute("""
            UPDATE bolsa SET
                cpf_doador = %s, data_coleta = %s, data_validade = %s,
                id_agendamento = %s, tipo_sanguineo = %s, status = %s
            WHERE cod_bolsa = %s
        """, (
            cpf_doador_bolsa.value, data_coleta_bolsa.value, data_validade_bolsa.value,
            id_agendamento_bolsa.value, tipo_sanguineo_bolsa.value, status_bolsa.value,
            cod_bolsa.value
        ))
        con.commit()
        if cursor.rowcount > 0:
            return pn.pane.Alert("Bolsa atualizada com sucesso!", alert_type="success")
        else:
            return pn.pane.Alert("Código da bolsa não encontrado.", alert_type="warning")
    except Exception as e:
        con.rollback()# Validação para garantir que os campos não estão vazios antes de inserir
        return pn.pane.Alert(f"Erro ao editar: {str(e)}", alert_type="danger")

def on_consultar_bolsa():
    if not cod_bolsa.value:
        return pn.pane.Alert("Digite o Código da Bolsa para consultar.", alert_type="warning")
    try:
        query = f"SELECT * FROM bolsa WHERE cod_bolsa = {cod_bolsa.value}"
        df = pd.read_sql_query(query, engine)
        if df.empty:
            return pn.pane.Alert("Bolsa não encontrada.", alert_type="warning")
        
        formatador_bolsa = {'cod_bolsa': {'type': 'number', 'thousand': ''}}
        
        return pn.widgets.Tabulator(df, show_index=False, formatters=formatador_bolsa)
    except Exception as e:
        return pn.pane.Alert(f"Erro na consulta: {str(e)}", alert_type="danger")


def render_bolsa(consultar, inserir, excluir, editar):
    if consultar:
        return on_consultar_bolsa()
    elif inserir:
        return on_inserir_bolsa()
    elif excluir:
        return on_excluir_bolsa()
    elif editar:
        return on_editar_bolsa()
    else:
        return queryAllBolsas()

tabela_bolsa = pn.bind(render_bolsa, btn_consultar_bolsa, btn_inserir_bolsa, btn_excluir_bolsa, btn_editar_bolsa)


tela_bolsas = pn.Row(
    pn.Column(
        "### Gerenciamento de Bolsas",
        cod_bolsa,
        cpf_doador_bolsa,
        tipo_sanguineo_bolsa,
        status_bolsa,
        data_coleta_bolsa,
        data_validade_bolsa,
        id_agendamento_bolsa,
        pn.Row(btn_consultar_bolsa, btn_inserir_bolsa, btn_editar_bolsa, btn_excluir_bolsa),
    ),
    pn.Column(
        tabela_bolsa
    )
)


In [19]:
tela_bolsas

Row
    [0] Column
        [0] Markdown(str)
        [1] IntInput(name='Código da Bolsa', placeholder='Digite o código...')
        [2] TextInput(name='CPF do Doador', placeholder='Digite o CPF...')
        [3] Select(name='Tipo Sanguíneo', options=['A+', 'A-', 'B+', ...], value='A+')
        [4] Select(name='Status da Bolsa', options=['Disponível', ...], value='Disponível')
        [5] DatePicker(name='Data da Coleta')
        [6] DatePicker(name='Data de Validade')
        [7] IntInput(name='ID do Agendamento', placeholder='Digite o ID...')
        [8] Row
            [0] Button(button_type='primary', name='Consultar', width=85)
            [1] Button(button_type='success', name='Inserir', width=85)
            [2] Button(button_type='warning', name='Editar', width=85)
            [3] Button(button_type='danger', name='Excluir', width=85)
    [1] Column
        [0] ParamFunction(function, _pane=Tabulator, defer_load=False)